# Height-Flatten 1x1 Block Training

This notebook uses generated `message.png` images with `fit_mode="height-flatten"`.
In this mode each `ImgConfig.img_size` block, `28x28` by default, is reduced to one `1x1` pixel before the pixels are reshaped into a square image.

Both the PyTorch and MLX conditional DDPM pipelines can be run from this notebook.

The notebook avoids live widget/rich-display image outputs. Preview and sample images are written as PNG files and their paths are printed, so reopening the notebook after a kernel restart does not leave stale widget state.

In [ ]:
from pathlib import Path
import json
import sys
from datetime import datetime

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

DATA_ROOT = REPO_ROOT / "data" / "images"
JSON_ROOT = REPO_ROOT / "output" / "json"
OUTPUT_ROOT = REPO_ROOT / "output"

def print_decode_comparison_summary(path: Path) -> None:
    path = Path(path)
    if not path.exists():
        print("decode_comparison_missing:", path)
        return
    payload = json.loads(path.read_text(encoding="utf-8"))
    records = payload.get("records", [])
    print("decode_comparison:", path)
    print("decode_total:", payload.get("total"))
    print("decode_comparable:", payload.get("comparable"))
    print("decode_matches:", payload.get("matches"))
    print("decode_all_match:", payload.get("all_match"))
    print("decoded_byte_comparable:", payload.get("decoded_byte_comparable"))
    print("decoded_byte_matches:", payload.get("decoded_byte_matches"))
    print("decoded_byte_all_match:", payload.get("decoded_byte_all_match"))
    if records:
        first = records[0]
        print("first_match:", first.get("match"))
        print("first_decoded_byte_hamming_distance_bits:", first.get("decoded_byte_hamming_distance_bits"))
        print("first_decoded_byte_hamming_distance_bytes:", first.get("decoded_byte_hamming_distance_bytes"))
        print("first_source_unique_rgb_color_count:", first.get("source", {}).get("unique_rgb_color_count"))
        print("first_final_unique_rgb_color_count:", first.get("final", {}).get("unique_rgb_color_count"))
        print("first_source_hex:", first.get("source", {}).get("hex"))
        print("first_final_hex:", first.get("final", {}).get("hex"))

def _canonical_hex(value: object) -> str | None:
    if not isinstance(value, str):
        return None
    text = value.strip().lower()
    return text if text.startswith("0x") else f"0x{text}"

def _hex_to_bytes(value: str | None) -> bytes | None:
    if value is None:
        return None
    try:
        return bytes.fromhex(value[2:] if value.startswith("0x") else value)
    except ValueError:
        return None

def _hamming_distance_hex(left: str | None, right: str | None) -> int | None:
    left_bytes = _hex_to_bytes(left)
    right_bytes = _hex_to_bytes(right)
    if left_bytes is None or right_bytes is None or len(left_bytes) != len(right_bytes):
        return None
    return sum((l_byte ^ r_byte).bit_count() for l_byte, r_byte in zip(left_bytes, right_bytes))

def _payload_message_hex(payload: dict[str, object]) -> str | None:
    message = payload.get("Message")
    if isinstance(message, dict):
        return _canonical_hex(message.get("Hex"))
    return _canonical_hex(message)

def _payload_final_hash(payload: dict[str, object]) -> str | None:
    for key in ("Generated hash", "Correct   hash", "Correct hash"):
        value = _canonical_hex(payload.get(key))
        if value is not None:
            return value
    return None

def _manifest_by_file(path: Path) -> dict[str, dict[str, object]]:
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    return {str(item.get("file")): item for item in payload if isinstance(item, dict)}

def _find_json_logs_for_conditions(json_root: Path, conditions: set[str]) -> dict[str, dict[str, object]]:
    targets = {_canonical_hex(condition): condition for condition in conditions if condition}
    targets = {key: value for key, value in targets.items() if key is not None}
    found: dict[str, dict[str, object]] = {}
    if not targets:
        return found
    remaining = set(targets)
    for path in sorted(Path(json_root).rglob("*.json")):
        if not remaining:
            break
        text = path.read_text(encoding="utf-8")
        hit = next((condition for condition in remaining if condition in text.lower()), None)
        if hit is None:
            continue
        payload = json.loads(text)
        final_hash = _payload_final_hash(payload)
        if final_hash == hit:
            found[targets[hit]] = {"path": str(path), "payload": payload}
            remaining.remove(hit)
    return found

def validate_sample_decoding_against_logs(
    decode_comparison_path: Path,
    json_root: Path = JSON_ROOT,
) -> dict[str, object] | None:
    from diffusion_hash_inv.models.sample_decoding import decode_sample_image

    decode_comparison_path = Path(decode_comparison_path)
    if not decode_comparison_path.exists():
        print("decode_validation_missing:", decode_comparison_path)
        return None

    sample_dir = decode_comparison_path.parent
    source_manifest_path = sample_dir / "source" / "source.labels.json"
    final_manifest_path = sample_dir / "final" / "final.labels.json"
    if not source_manifest_path.exists():
        source_manifest_path = next((sample_dir / "source").glob("*.labels.json"), None)
    if not final_manifest_path.exists():
        final_manifest_path = next((sample_dir / "final").glob("*.labels.json"), None)
    if source_manifest_path is None or final_manifest_path is None or not source_manifest_path.exists() or not final_manifest_path.exists():
        print("decode_validation_manifest_missing:", source_manifest_path, final_manifest_path)
        return None

    decode_payload = json.loads(decode_comparison_path.read_text(encoding="utf-8"))
    source_manifest = _manifest_by_file(source_manifest_path)
    final_manifest = _manifest_by_file(final_manifest_path)
    conditions = {
        str(item.get("condition"))
        for item in [*source_manifest.values(), *final_manifest.values()]
        if item.get("condition") is not None
    }
    json_logs = _find_json_logs_for_conditions(json_root, conditions)

    records = []
    for record in decode_payload.get("records", []):
        source_path = Path(record["source_file"])
        final_path = Path(record["final_file"])
        source_meta = source_manifest.get(source_path.name, {})
        final_meta = final_manifest.get(final_path.name, {})
        source_condition = str(source_meta.get("condition"))
        final_condition = str(final_meta.get("condition"))
        condition = source_condition if source_condition != "None" else final_condition
        log_entry = json_logs.get(condition)
        log_payload = log_entry["payload"] if log_entry is not None else {}
        expected_message_hex = _payload_message_hex(log_payload)
        expected_final_hash = _payload_final_hash(log_payload)

        source_decoded = decode_sample_image(source_path)
        final_decoded = decode_sample_image(final_path)
        source_hex = _canonical_hex(source_decoded.get("hex"))
        final_hex = _canonical_hex(final_decoded.get("hex"))
        source_complete = bool(source_decoded.get("supported")) and bool(source_decoded.get("complete"))
        final_complete = bool(final_decoded.get("supported")) and bool(final_decoded.get("complete"))
        source_final_equal_hex = source_hex is not None and source_hex == final_hex
        records.append(
            {
                "index": record.get("index"),
                "source_file": str(source_path),
                "final_file": str(final_path),
                "source_condition": source_condition,
                "final_condition": final_condition,
                "conditions_equal": source_condition == final_condition,
                "json_log": None if log_entry is None else log_entry["path"],
                "expected_message_hex": expected_message_hex,
                "expected_final_hash": expected_final_hash,
                "log_hash_matches_condition": expected_final_hash == _canonical_hex(condition),
                "source_supported": bool(source_decoded.get("supported")),
                "source_complete": source_complete,
                "source_hex": source_hex,
                "source_matches_log_message": source_complete and source_hex == expected_message_hex,
                "final_supported": bool(final_decoded.get("supported")),
                "final_complete": final_complete,
                "final_hex": final_hex,
                "final_matches_log_message": final_complete and final_hex == expected_message_hex,
                "source_final_equal_hex": source_final_equal_hex,
                "source_final_complete_equal": source_complete and final_complete and source_final_equal_hex,
                "hamming_distance_bits": _hamming_distance_hex(source_hex, final_hex),
            }
        )

    summary = {
        "total": len(records),
        "json_logs_found": sum(record["json_log"] is not None for record in records),
        "source_final_complete_equal": sum(record["source_final_complete_equal"] for record in records),
        "source_matches_log_message": sum(record["source_matches_log_message"] for record in records),
        "final_matches_log_message": sum(record["final_matches_log_message"] for record in records),
        "log_hash_matches_condition": sum(record["log_hash_matches_condition"] for record in records),
        "all_source_final_complete_equal": bool(records) and all(record["source_final_complete_equal"] for record in records),
        "all_source_matches_log_message": bool(records) and all(record["source_matches_log_message"] for record in records),
        "all_final_matches_log_message": bool(records) and all(record["final_matches_log_message"] for record in records),
    }
    validation = {"summary": summary, "records": records}
    validation_path = decode_comparison_path.with_name("decode_log_validation.json")
    validation_path.write_text(json.dumps(validation, indent=2), encoding="utf-8")
    print("decode_log_validation:", validation_path)
    print("log_validation_summary:", summary)
    if records:
        first = records[0]
        print("first_expected_message_hex:", first["expected_message_hex"])
        print("first_source_matches_log_message:", first["source_matches_log_message"])
        print("first_final_matches_log_message:", first["final_matches_log_message"])
        print("first_source_final_complete_equal:", first["source_final_complete_equal"])
    return validation

FIT_MODE = "height-flatten"
CHANNELS = 3
IMAGE_SIZE = 64  # ignored by height-flatten; kept for pad/resize compatibility
MAX_IMAGES = None
SEED = 0

TRAIN_STEPS = 50_000
BETA_SCHEDULE = "linear"  # linear, hash-approach1, hash-approach2
DIFFUSION_TIMESTEPS = "auto"  # integer or auto for linear schedule
BETA_SCHEDULE_STEP = "4th Step"
BETA_START = 1e-4
BETA_END = 2e-2
BATCH_SIZE = 256
LEARNING_RATE = 2e-4
LOG_EVERY = 100
CHECKPOINT_EVERY = 1_000
SAMPLE_EVERY = 5_000
SAMPLE_COUNT = 16
SAMPLE_COLUMNS = 4
SAVE_PROCESS_TRACES = True
TRACE_SAMPLE_COUNT = 4

TORCH_OUTPUT_DIR = OUTPUT_ROOT / f"diffusion_torch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
TORCH_DEVICE = "auto"
TORCH_BASE_CHANNELS = 64
TORCH_TIME_DIM = 256
TORCH_NUM_WORKERS = 0
TORCH_SAVE_TRAIN_BATCHES_EVERY = 0

MLX_OUTPUT_DIR = OUTPUT_ROOT / f"diffusion_mlx_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
MLX_DEVICE = "gpu"
MLX_TIME_DIM = 64
MLX_HIDDEN_DIM = 256

RUN_TORCH_TRAIN = True
RUN_MLX_TRAIN = True

print("repo:", REPO_ROOT)
print("data:", DATA_ROOT)
print("json:", JSON_ROOT)
print("fit_mode:", FIT_MODE)
print("max_images:", MAX_IMAGES)
print("train_steps:", TRAIN_STEPS)
print("beta_schedule:", BETA_SCHEDULE)
print("diffusion_timesteps:", DIFFUSION_TIMESTEPS)
print("beta_schedule_step:", BETA_SCHEDULE_STEP)
print("batch_size:", BATCH_SIZE)
print("checkpoint_every:", CHECKPOINT_EVERY)
print("save_process_traces:", SAVE_PROCESS_TRACES)
print("trace_sample_count:", TRACE_SAMPLE_COUNT)
print("torch_output:", TORCH_OUTPUT_DIR)
print("mlx_output:", MLX_OUTPUT_DIR)

In [ ]:
import numpy as np
from PIL import Image

from diffusion_hash_inv.config import ImgConfig
from diffusion_hash_inv.models.conditional_diffusion import _fit_image as torch_fit_image

PREVIEW_DIR = OUTPUT_ROOT / "notebook_previews"
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

def first_message_path(data_root: Path) -> Path | None:
    return next(data_root.rglob("message.png"), None) if data_root.exists() else None

def pixel_grid_preview(image: Image.Image, scale: int = 16, max_size: int = 256) -> Image.Image:
    # Expand each 1x1 fitted pixel into a solid scale x scale block without interpolation.
    max_dim = max(image.size)
    scale_limit = max(1, max_size // max_dim)
    scale = max(1, min(scale, scale_limit))
    arr = np.asarray(image, dtype=np.uint8)
    expanded = np.repeat(np.repeat(arr, scale, axis=0), scale, axis=1)
    return Image.fromarray(expanded)

def preview_height_flatten(path: Path, channels: int = CHANNELS) -> tuple[Path, Path]:
    with Image.open(path) as source:
        fitted = torch_fit_image(source, IMAGE_SIZE, FIT_MODE, channels)
        fitted_path = PREVIEW_DIR / "height_flatten_model_input.png"
        fitted.save(fitted_path)

        preview = pixel_grid_preview(fitted, scale=16, max_size=256)
        preview_path = PREVIEW_DIR / "height_flatten_preview.png"
        preview.save(preview_path)

        print("source:", source.size, "ImgConfig.img_size:", ImgConfig().img_size)
        print("height-flatten model input:", fitted.size, "channels:", channels)
        print("model_input_png:", fitted_path)
        print("preview_png:", preview_path, "preview_size:", preview.size)
        return fitted_path, preview_path

sample_path = first_message_path(DATA_ROOT)
if sample_path is None:
    print("No message.png found. Generate hash images first or update DATA_ROOT.")
else:
    print("sample:", sample_path)
    preview_height_flatten(sample_path)


## Sample Result Saving

PyTorch and MLX both save sample artifacts as per-sample `source_*.png` and `final_*.png` files.

Final samples are saved under `output_dir / "sample"`:

- `source/source_*.png`: matched source samples from the dataset.
- `final/final_*.png`: generated samples from the reverse process.
- `source/source.labels.json`, `final/final.labels.json`: sample manifests.
- `decode_comparison.json`: source/final RGB colors, Byte2RGB decoded byte comparison, and decoded-byte hamming distance; the training cells print its match summary.
- `decode_log_validation.json`: fresh source/final decoding checked against the matching JSON log `Message.Hex` and final hash.

Intermediate samples from `SAMPLE_EVERY` are saved under `sample/step_XXXXXX/` with the same source/final folder layout.

When `SAVE_PROCESS_TRACES` is enabled, both backends save every forward and reverse diffusion step under `process_traces/forward` and `process_traces/reverse`.

## PyTorch Conditional DDPM

The PyTorch config below is set for a full height-flatten experiment. `RUN_TORCH_TRAIN` is `False` by default to avoid accidentally starting a long notebook run; set it to `True` in the config cell when you are ready to train.

In [ ]:
import importlib
import diffusion_hash_inv.models.conditional_diffusion as conditional_diffusion_module

conditional_diffusion_module = importlib.reload(conditional_diffusion_module)
ConditionalDiffusionTrainConfig = conditional_diffusion_module.ConditionalDiffusionTrainConfig
GeneratedImageDataset = conditional_diffusion_module.GeneratedImageDataset
train_conditional_diffusion = conditional_diffusion_module.train_conditional_diffusion

torch_dataset = None
if DATA_ROOT.exists() and JSON_ROOT.exists():
    torch_dataset = GeneratedImageDataset(
        DATA_ROOT,
        json_root=JSON_ROOT,
        image_size=IMAGE_SIZE,
        channels=CHANNELS,
        fit_mode=FIT_MODE,
        max_images=MAX_IMAGES,
    )
    image, label, loop_meta = torch_dataset[0]
    print("dataset_size:", len(torch_dataset))
    print("sample_tensor_shape:", tuple(image.shape), "label:", int(label))
else:
    print("DATA_ROOT or JSON_ROOT does not exist yet.")

In [ ]:
torch_result = None

torch_config = ConditionalDiffusionTrainConfig(
    data_root=DATA_ROOT,
    json_root=JSON_ROOT,
    output_dir=TORCH_OUTPUT_DIR,
    image_size=IMAGE_SIZE,
    channels=CHANNELS,
    fit_mode=FIT_MODE,
    max_images=MAX_IMAGES,
    batch_size=BATCH_SIZE,
    train_steps=TRAIN_STEPS,
    timesteps=DIFFUSION_TIMESTEPS,
    learning_rate=LEARNING_RATE,
    beta_start=BETA_START,
    beta_end=BETA_END,
    beta_schedule=BETA_SCHEDULE,
    beta_schedule_step=BETA_SCHEDULE_STEP,
    base_channels=TORCH_BASE_CHANNELS,
    time_dim=TORCH_TIME_DIM,
    sample_count=SAMPLE_COUNT,
    sample_every=SAMPLE_EVERY,
    save_process_traces=SAVE_PROCESS_TRACES,
    trace_sample_count=TRACE_SAMPLE_COUNT,
    checkpoint_every=CHECKPOINT_EVERY,
    save_train_batches_every=TORCH_SAVE_TRAIN_BATCHES_EVERY,
    num_workers=TORCH_NUM_WORKERS,
    device=TORCH_DEVICE,
    seed=SEED,
    log_every=LOG_EVERY,
)

if RUN_TORCH_TRAIN:
    torch_result = train_conditional_diffusion(torch_config)
    print("checkpoint:", torch_result["checkpoint"])
    print("result_keys:", sorted(torch_result))
    print("sample_dir:", torch_result["sample_dir"])
    print("sample_final_dir:", torch_result["sample_final_dir"])
    print("sample_source_dir:", torch_result["sample_source_dir"])
    print("sample_final_manifest:", torch_result["sample_final_manifest"])
    print("sample_source_manifest:", torch_result["sample_source_manifest"])
    print("sample_decode_comparison:", torch_result["sample_decode_comparison"])
    print_decode_comparison_summary(torch_result["sample_decode_comparison"])
    validate_sample_decoding_against_logs(torch_result["sample_decode_comparison"])
    print("sample_final_files:", len(torch_result["sample_final_files"]))
    print("sample_source_files:", len(torch_result["sample_source_files"]))
    print("process_traces:", torch_result.get("process_traces"))
else:
    print("Torch config ready:", torch_config.output_dir)
    print("expected final checkpoint:", torch_config.output_dir / "checkpoints" / f"step_{TRAIN_STEPS:06d}.pt")
    print("expected sample_dir:", torch_config.output_dir / "sample")
    print("expected final_manifest:", torch_config.output_dir / "sample" / "final" / "final.labels.json")
    print("expected source_manifest:", torch_config.output_dir / "sample" / "source" / "source.labels.json")
    print("expected decode_comparison:", torch_config.output_dir / "sample" / "decode_comparison.json")
    print("expected process_traces:", torch_config.output_dir / "process_traces")

## MLX Conditional DDPM

The MLX config mirrors the same data, diffusion schedule, checkpoint cadence, and sample count. `RUN_MLX_TRAIN` is `False` by default to avoid accidentally starting a long notebook run. Use `MLX_DEVICE = "gpu"` only on a machine where Metal is available.

In [ ]:
try:
    import importlib
    import mlx.core as mx
    import diffusion_hash_inv.models.conditional_diffusion_mlx as conditional_diffusion_mlx_module

    conditional_diffusion_mlx_module = importlib.reload(conditional_diffusion_mlx_module)
    MLXConditionalDiffusionTrainConfig = (
        conditional_diffusion_mlx_module.MLXConditionalDiffusionTrainConfig
    )
    MLXGeneratedImageDataset = conditional_diffusion_mlx_module.MLXGeneratedImageDataset
    train_conditional_diffusion_mlx = conditional_diffusion_mlx_module.train_conditional_diffusion_mlx
    MLX_AVAILABLE = True
except Exception as exc:
    MLX_AVAILABLE = False
    print("MLX is not available:", exc)

mlx_dataset = None
if MLX_AVAILABLE and DATA_ROOT.exists() and JSON_ROOT.exists():
    mlx_dataset = MLXGeneratedImageDataset(
        DATA_ROOT,
        json_root=JSON_ROOT,
        image_size=IMAGE_SIZE,
        channels=CHANNELS,
        fit_mode=FIT_MODE,
        max_images=MAX_IMAGES,
        seed=SEED,
    )
    image, label = mlx_dataset[0]
    print("dataset_size:", len(mlx_dataset))
    print("output_image_size:", mlx_dataset.output_image_size)
    print("image_dim:", mlx_dataset.image_dim, "sample_vector_shape:", image.shape, "label:", label)
elif MLX_AVAILABLE:
    print("DATA_ROOT or JSON_ROOT does not exist yet.")

In [ ]:
mlx_sample_path = None

if MLX_AVAILABLE:
    mlx_config = MLXConditionalDiffusionTrainConfig(
        data_root=DATA_ROOT,
        json_root=JSON_ROOT,
        output_dir=MLX_OUTPUT_DIR,
        image_size=IMAGE_SIZE,
        channels=CHANNELS,
        fit_mode=FIT_MODE,
        max_images=MAX_IMAGES,
        batch_size=BATCH_SIZE,
        train_steps=TRAIN_STEPS,
        timesteps=DIFFUSION_TIMESTEPS,
        learning_rate=LEARNING_RATE,
        beta_start=BETA_START,
        beta_end=BETA_END,
        beta_schedule=BETA_SCHEDULE,
        beta_schedule_step=BETA_SCHEDULE_STEP,
        time_dim=MLX_TIME_DIM,
        hidden_dim=MLX_HIDDEN_DIM,
        sample_count=SAMPLE_COUNT,
        sample_every=SAMPLE_EVERY,
        save_process_traces=SAVE_PROCESS_TRACES,
        trace_sample_count=TRACE_SAMPLE_COUNT,
        columns=SAMPLE_COLUMNS,
        device=MLX_DEVICE,
        seed=SEED,
        log_every=LOG_EVERY,
        checkpoint_every=CHECKPOINT_EVERY,
    )
    if RUN_MLX_TRAIN:
        mlx_sample_path = train_conditional_diffusion_mlx(mlx_config)
        mlx_sample_dir = mlx_sample_path.parent.parent
        print("sample_dir:", mlx_sample_dir)
        print("sample_final_dir:", mlx_sample_dir / "final")
        print("sample_source_dir:", mlx_sample_dir / "source")
        print("sample_final_manifest:", mlx_sample_path)
        print("sample_source_manifest:", (mlx_sample_dir / "source" / mlx_config.source_name).with_suffix(".labels.json"))
        print("sample_decode_comparison:", mlx_sample_dir / "decode_comparison.json")
        print_decode_comparison_summary(mlx_sample_dir / "decode_comparison.json")
        validate_sample_decoding_against_logs(mlx_sample_dir / "decode_comparison.json")
        print("final_checkpoint:", mlx_config.output_dir / "checkpoints" / f"step_{TRAIN_STEPS:06d}.json")
        print("process_traces:", mlx_config.output_dir / "process_traces")
    else:
        print("MLX config ready:", mlx_config.output_dir)
        print("expected final_checkpoint:", mlx_config.output_dir / "checkpoints" / f"step_{TRAIN_STEPS:06d}.json")
        print("expected sample_dir:", mlx_config.output_dir / "sample")
        print("expected final_manifest:", (mlx_config.output_dir / "sample" / "final" / mlx_config.sample_name).with_suffix(".labels.json"))
        print("expected source_manifest:", (mlx_config.output_dir / "sample" / "source" / mlx_config.source_name).with_suffix(".labels.json"))
        print("expected decode_comparison:", mlx_config.output_dir / "sample" / "decode_comparison.json")
        print("expected process_traces:", mlx_config.output_dir / "process_traces")